# Step 4: Optimized Pipeline

After analyzing Step 3 results, we found two key issues that limited accuracy:

1. The original cascade was using fuzzy match too aggressively (threshold 60). Fuzzy match at threshold 60 has only ~38% precision on the matches it produces. Letting it dominate meant we threw away cases where the classifier or retrieval would have been more accurate.

2. The Logistic Regression classifier was undertrained. By switching to Linear SVM with lower regularization (C=2.0) and lowering the minimum-examples filter from 3 to 2, classifier accuracy jumped from 27.2% to 33.0% in isolation.

**This notebook implements the improved pipeline:**
- Better Linear SVM classifier (replaces the Logistic Regression in Step 3)
- Weighted-confidence ensemble that picks the best method per-literal instead of strict cascade order
- All thresholds tuned on the validation set

Files to upload:
- `train_preprocessed.csv` (from Step 0)
- `test_preprocessed.csv` (from Step 0)
- `reference_preprocessed.csv` (from Step 0)
- `step2_predictions.csv` (from Step 2 — provides fuzzy and retrieval results)

Files produced:
- `kaggle_submission_optimized.csv` (final submission file)
- `final_predictions_optimized.csv` (full detail)

## 1. Imports and loading

In [1]:
!pip install rapidfuzz -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from rapidfuzz import fuzz, process
from scipy.special import expit
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train_preprocessed.csv')
test = pd.read_csv('test_preprocessed.csv')
reference = pd.read_csv('reference_preprocessed.csv')
step2 = pd.read_csv('step2_predictions.csv')

# Defensive: fill any NaN strings
train['Literal_norm'] = train['Literal_norm'].fillna('')
test['Literal_norm'] = test['Literal_norm'].fillna('')
reference['Description_norm'] = reference['Description_norm'].fillna('')

print(f'Training: {len(train):,} rows')
print(f'Test:     {len(test):,} rows')
print(f'Step 2 predictions: {len(step2):,} rows')

Training: 13,700 rows
Test:     6,667 rows
Step 2 predictions: 6,667 rows


## 2. Train improved Linear SVM classifiers

The improvements over Step 3:
- LinearSVC instead of Logistic Regression. LinearSVC trains faster and gives slightly better accuracy on text classification.
- C=2.0 instead of C=1.0. This is less regularization, letting the model fit the training data more tightly. With this much vocabulary, we want stronger feature weights.
- MIN_EXAMPLES=2 instead of 3. This includes more codes in training (covers more of the long tail) at a small risk of overfitting on each.
- We keep two separate classifiers for diagnosis and procedure plus the router.

In [4]:
MIN_EXAMPLES = 2
C_VALUE = 2.0

def filter_by_min(df, min_count):
    counts = df['Code'].value_counts()
    return df[df['Code'].isin(counts[counts >= min_count].index)].copy()

train_diag = filter_by_min(train[train.code_type == 'diagnosis'], MIN_EXAMPLES)
train_proc = filter_by_min(train[train.code_type == 'procedure'], MIN_EXAMPLES)

print(f'After filter (min {MIN_EXAMPLES} examples):')
print(f'  Diagnosis: {len(train_diag):,} rows, {train_diag.Code.nunique():,} unique codes')
print(f'  Procedure: {len(train_proc):,} rows, {train_proc.Code.nunique():,} unique codes')

After filter (min 2 examples):
  Diagnosis: 7,703 rows, 1,251 unique codes
  Procedure: 4,082 rows, 893 unique codes


In [5]:
def build_features():
    """Combined char + word TF-IDF feature extractor."""
    return FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2, sublinear_tf=True)),
        ('word', TfidfVectorizer(analyzer='word',    ngram_range=(1,2), min_df=2, sublinear_tf=True)),
    ])

print('Training diagnosis SVM...')
start = time.time()
vec_diag = build_features()
Xd = vec_diag.fit_transform(train_diag['Literal_norm'])
clf_diag = LinearSVC(C=C_VALUE, class_weight='balanced', max_iter=3000, random_state=42)
clf_diag.fit(Xd, train_diag['Code'].values)
print(f'  Done in {time.time()-start:.1f}s, {len(clf_diag.classes_)} classes, features {Xd.shape[1]}')

print('Training procedure SVM...')
start = time.time()
vec_proc = build_features()
Xp = vec_proc.fit_transform(train_proc['Literal_norm'])
clf_proc = LinearSVC(C=C_VALUE, class_weight='balanced', max_iter=3000, random_state=42)
clf_proc.fit(Xp, train_proc['Code'].values)
print(f'  Done in {time.time()-start:.1f}s, {len(clf_proc.classes_)} classes, features {Xp.shape[1]}')

print('Training diagnosis/procedure router...')
start = time.time()
vec_router = build_features()
Xr = vec_router.fit_transform(train['Literal_norm'])
clf_router = LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, random_state=42)
clf_router.fit(Xr, train['code_type'].values)
print(f'  Done in {time.time()-start:.1f}s')

Training diagnosis SVM...
  Done in 17.7s, 1251 classes, features 18474
Training procedure SVM...
  Done in 6.5s, 893 classes, features 12608
Training diagnosis/procedure router...
  Done in 0.4s


## 3. Apply classifiers to the test set

We use `decision_function` instead of `predict_proba` because LinearSVC doesn't provide probability scores directly. Decision values are signed numbers — positive means the model thinks this class is likely, negative means unlikely. We later transform them with a sigmoid to get something in [0,1] for the ensemble.

In [6]:
test['router_pred'] = clf_router.predict(vec_router.transform(test['Literal_norm']))
print('Router predictions:', test['router_pred'].value_counts().to_dict())

Router predictions: {'diagnosis': 4037, 'procedure': 2630}


In [7]:
diag_idx = test[test['router_pred'] == 'diagnosis'].index
proc_idx = test[test['router_pred'] == 'procedure'].index

test['clf_code'] = ''
test['clf_decision'] = 0.0

if len(diag_idx):
    Xvd = vec_diag.transform(test.loc[diag_idx, 'Literal_norm'])
    decisions = clf_diag.decision_function(Xvd)
    test.loc[diag_idx, 'clf_code'] = clf_diag.classes_[decisions.argmax(axis=1)]
    test.loc[diag_idx, 'clf_decision'] = decisions.max(axis=1)

if len(proc_idx):
    Xvp = vec_proc.transform(test.loc[proc_idx, 'Literal_norm'])
    decisions = clf_proc.decision_function(Xvp)
    test.loc[proc_idx, 'clf_code'] = clf_proc.classes_[decisions.argmax(axis=1)]
    test.loc[proc_idx, 'clf_decision'] = decisions.max(axis=1)

# Transform decision values to confidence-like scores using sigmoid
test['clf_confidence'] = expit(test['clf_decision'])

print('Classifier confidence distribution:')
print(test['clf_confidence'].describe())

Classifier confidence distribution:
count    6667.000000
mean        0.540228
std         0.126314
min         0.277567
25%         0.431295
50%         0.547116
75%         0.655091
max         0.853668
Name: clf_confidence, dtype: float64


## 4. Merge with Step 2 results

We bring in the fuzzy and retrieval scores that were computed in earlier notebooks. We need everything in one place for the ensemble.

In [8]:
# Bring in step 1 codes and step 2 retrieval
test = test.merge(
    step2[['id', 'step1_code', 'step1_method', 'retrieval_code', 'retrieval_score']],
    on='id', how='left'
)

# We also need the fuzzy score per literal. Rebuild it here to be safe.
print('Rebuilding fuzzy scores for completeness...')
start = time.time()
exact_lookup = {}
for lit_norm, group in train.groupby('Literal_norm'):
    exact_lookup[lit_norm] = group['Code'].value_counts().index[0]

test['exact_code'] = test['Literal_norm'].map(exact_lookup)
test['fuzzy_code'] = test['exact_code'].copy()
test['fuzzy_score'] = test['exact_code'].apply(lambda x: 100.0 if pd.notna(x) else 0.0)

lookup_keys = list(exact_lookup.keys())
for idx, row in test[test['exact_code'].isna()].iterrows():
    r = process.extractOne(row['Literal_norm'], lookup_keys, scorer=fuzz.ratio, score_cutoff=0)
    if r is not None:
        test.loc[idx, 'fuzzy_code'] = exact_lookup[r[0]]
        test.loc[idx, 'fuzzy_score'] = r[1]

print(f'Done in {time.time()-start:.1f}s')

Rebuilding fuzzy scores for completeness...
Done in 3.8s


## 5. Apply the optimized ensemble strategy

For each test literal we have four candidate predictions, each with a confidence score:
- **Exact match** (confidence 1.0 if matched, 0 otherwise)
- **Fuzzy match** (confidence = fuzzy_score / 100)
- **Retrieval** (confidence = cosine similarity score, 0-1)
- **Classifier** (confidence = sigmoid of SVM decision value, 0-1)

Strategy:
1. If exact match exists, use it (highest confidence by design)
2. Otherwise, take a weighted vote between the three remaining methods, with weights tuned on the validation set:
   - Fuzzy weight: 1.1
   - Retrieval weight: 1.0
   - Classifier weight: 1.5

In [9]:
FUZZY_WEIGHT = 1.1
RETRIEVAL_WEIGHT = 1.0
CLASSIFIER_WEIGHT = 1.5

def predict_one(row):
    """Pick the final prediction for one row using weighted confidence."""
    # Exact match is always taken when available
    if pd.notna(row['exact_code']) and row['exact_code'] != '':
        return row['exact_code'], 'exact', 1.0
    
    # Otherwise compare the other three methods
    candidates = []
    
    # Fuzzy match
    if pd.notna(row['fuzzy_code']) and row['fuzzy_score'] > 0:
        candidates.append((row['fuzzy_score'] / 100.0 * FUZZY_WEIGHT, row['fuzzy_code'], 'fuzzy'))
    
    # Retrieval
    if pd.notna(row['retrieval_code']):
        candidates.append((row['retrieval_score'] * RETRIEVAL_WEIGHT, row['retrieval_code'], 'retrieval'))
    
    # Classifier
    if pd.notna(row['clf_code']) and row['clf_code'] != '':
        candidates.append((row['clf_confidence'] * CLASSIFIER_WEIGHT, row['clf_code'], 'classifier'))
    
    if not candidates:
        return '', 'none', 0.0
    
    best = max(candidates, key=lambda x: x[0])
    return best[1], best[2], best[0]

In [10]:
print('Applying ensemble strategy to test set...')
results = test.apply(predict_one, axis=1)
test['final_code'] = [r[0] for r in results]
test['final_method'] = [r[1] for r in results]
test['final_confidence'] = [r[2] for r in results]

print('Final method usage:')
print(test['final_method'].value_counts())
print()
print(f'Total predictions: {(test["final_code"] != "").sum():,} / {len(test):,}')

Applying ensemble strategy to test set...
Final method usage:
final_method
exact         3812
fuzzy         2118
classifier     601
retrieval      136
Name: count, dtype: int64

Total predictions: 6,667 / 6,667


## 6. Validate the optimized pipeline

Same train/val split as before so we can compare against the original Step 3 results.

In [11]:
# Split the same way for fair comparison
tr_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
tr_split = tr_split.copy()
val_split = val_split.copy()
tr_split['Literal_norm'] = tr_split['Literal_norm'].fillna('')
val_split['Literal_norm'] = val_split['Literal_norm'].fillna('')
n = len(val_split)
print(f'Validation set: {n:,}')

Validation set: 2,740


In [12]:
# Train validation classifiers (using only tr_split, so the evaluation on val_split is honest)
print('Training validation classifiers...')
start = time.time()

tr_diag_v = filter_by_min(tr_split[tr_split.code_type == 'diagnosis'], MIN_EXAMPLES)
tr_proc_v = filter_by_min(tr_split[tr_split.code_type == 'procedure'], MIN_EXAMPLES)

v_vec_d = build_features()
Xvd_train = v_vec_d.fit_transform(tr_diag_v['Literal_norm'])
v_clf_d = LinearSVC(C=C_VALUE, class_weight='balanced', max_iter=3000, random_state=42)
v_clf_d.fit(Xvd_train, tr_diag_v['Code'].values)

v_vec_p = build_features()
Xvp_train = v_vec_p.fit_transform(tr_proc_v['Literal_norm'])
v_clf_p = LinearSVC(C=C_VALUE, class_weight='balanced', max_iter=3000, random_state=42)
v_clf_p.fit(Xvp_train, tr_proc_v['Code'].values)

v_vec_r = build_features()
Xvr_train = v_vec_r.fit_transform(tr_split['Literal_norm'])
v_clf_r = LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, random_state=42)
v_clf_r.fit(Xvr_train, tr_split['code_type'].values)

print(f'Trained validation classifiers in {time.time()-start:.1f}s')

Training validation classifiers...
Trained validation classifiers in 16.2s


In [13]:
# Compute all four methods on validation set
print('Computing all 4 methods on validation set...')

# Step 1 lookup
v_exact_lookup = {}
for lit_norm, group in tr_split.groupby('Literal_norm'):
    v_exact_lookup[lit_norm] = group['Code'].value_counts().index[0]

val_split['exact_code'] = val_split['Literal_norm'].map(v_exact_lookup)
val_split['fuzzy_code'] = val_split['exact_code'].copy()
val_split['fuzzy_score'] = val_split['exact_code'].apply(lambda x: 100.0 if pd.notna(x) else 0.0)

v_keys = list(v_exact_lookup.keys())
for idx, row in val_split[val_split['exact_code'].isna()].iterrows():
    r = process.extractOne(row['Literal_norm'], v_keys, scorer=fuzz.ratio, score_cutoff=0)
    if r is not None:
        val_split.loc[idx, 'fuzzy_code'] = v_exact_lookup[r[0]]
        val_split.loc[idx, 'fuzzy_score'] = r[1]

print('  Step 1 (matching) done')

Computing all 4 methods on validation set...
  Step 1 (matching) done


In [14]:
# Step 2 retrieval on validation
print('Building retrieval index...')
rcv = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2, max_df=0.95, sublinear_tf=True)
rcr = rcv.fit_transform(reference['Description_norm'])
rwv = TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)
rwr = rwv.fit_transform(reference['Description_norm'])

print('Retrieval on validation set (takes 1-2 minutes)...')
start = time.time()
vc = rcv.transform(val_split['Literal_norm'])
vw = rwv.transform(val_split['Literal_norm'])

ret_codes, ret_scores = [], []
for i in range(0, len(val_split), 100):
    end = min(i + 100, len(val_split))
    cs = cosine_similarity(vc[i:end], rcr)
    ws = cosine_similarity(vw[i:end], rwr)
    comb = (cs + ws) / 2
    best = comb.argmax(axis=1)
    for j in range(end - i):
        bi = best[j]
        ret_codes.append(reference.iloc[bi]['Code'])
        ret_scores.append(float(comb[j, bi]))
    del cs, ws, comb

val_split['retrieval_code'] = ret_codes
val_split['retrieval_score'] = ret_scores
print(f'  Done in {time.time()-start:.1f}s')

Building retrieval index...
Retrieval on validation set (takes 1-2 minutes)...
  Done in 15.6s


In [15]:
# Step 3 classifier on validation
print('Applying validation classifier...')
val_split['router_pred'] = v_clf_r.predict(v_vec_r.transform(val_split['Literal_norm']))

vd_idx = val_split[val_split.router_pred == 'diagnosis'].index
vp_idx = val_split[val_split.router_pred == 'procedure'].index

val_split['clf_code'] = ''
val_split['clf_decision'] = 0.0

if len(vd_idx):
    Xvd = v_vec_d.transform(val_split.loc[vd_idx, 'Literal_norm'])
    dec = v_clf_d.decision_function(Xvd)
    val_split.loc[vd_idx, 'clf_code'] = v_clf_d.classes_[dec.argmax(axis=1)]
    val_split.loc[vd_idx, 'clf_decision'] = dec.max(axis=1)

if len(vp_idx):
    Xvp = v_vec_p.transform(val_split.loc[vp_idx, 'Literal_norm'])
    dec = v_clf_p.decision_function(Xvp)
    val_split.loc[vp_idx, 'clf_code'] = v_clf_p.classes_[dec.argmax(axis=1)]
    val_split.loc[vp_idx, 'clf_decision'] = dec.max(axis=1)

val_split['clf_confidence'] = expit(val_split['clf_decision'])

Applying validation classifier...


In [16]:
# Apply the same ensemble logic
results = val_split.apply(predict_one, axis=1)
val_split['final_code'] = [r[0] for r in results]
val_split['final_correct'] = (val_split['final_code'] == val_split['Code'])

# Also individual method accuracy
val_split['s1_correct'] = (val_split['exact_code'].fillna(val_split['fuzzy_code']) == val_split['Code'])
val_split['s2_correct'] = (val_split['retrieval_code'] == val_split['Code'])
val_split['s3_correct'] = (val_split['clf_code'] == val_split['Code'])

print('Validation accuracy:')
print(f'{"":40s} {"correct":>10s} {"accuracy":>10s}')
print('-' * 65)
print(f'{"Step 1 (exact + fuzzy)":40s} {val_split.s1_correct.sum():>10,} {val_split.s1_correct.mean()*100:>9.2f}%')
print(f'{"Step 2 (retrieval)":40s} {val_split.s2_correct.sum():>10,} {val_split.s2_correct.mean()*100:>9.2f}%')
print(f'{"Step 3 (Linear SVM classifier)":40s} {val_split.s3_correct.sum():>10,} {val_split.s3_correct.mean()*100:>9.2f}%')
print(f'{"Optimized ensemble (final)":40s} {val_split.final_correct.sum():>10,} {val_split.final_correct.mean()*100:>9.2f}%')

Validation accuracy:
                                            correct   accuracy
-----------------------------------------------------------------
Step 1 (exact + fuzzy)                        1,040     37.96%
Step 2 (retrieval)                              294     10.73%
Step 3 (Linear SVM classifier)                  903     32.96%
Optimized ensemble (final)                    1,071     39.09%


In [17]:
# How complementary are the methods?
s1 = val_split['s1_correct'].fillna(False)
s2 = val_split['s2_correct'].fillna(False)
s3 = val_split['s3_correct'].fillna(False)

print('Method overlap on validation set:')
print(f'  All three correct:     {(s1 & s2 & s3).sum():>5,}')
print(f'  Only Step 1 correct:    {(s1 & ~s2 & ~s3).sum():>5,}  (Step 1 unique value)')
print(f'  Only Step 2 correct:    {(~s1 & s2 & ~s3).sum():>5,}  (Step 2 unique value)')
print(f'  Only Step 3 correct:    {(~s1 & ~s2 & s3).sum():>5,}  (Step 3 unique value)')
print(f'  All three wrong:        {(~s1 & ~s2 & ~s3).sum():>5,}')
print(f'  Theoretical ceiling:    {(s1 | s2 | s3).sum():>5,}  ({(s1 | s2 | s3).mean()*100:.1f}%)')

Method overlap on validation set:
  All three correct:       105
  Only Step 1 correct:      284  (Step 1 unique value)
  Only Step 2 correct:      103  (Step 2 unique value)
  Only Step 3 correct:      165  (Step 3 unique value)
  All three wrong:        1,398
  Theoretical ceiling:    1,342  (49.0%)


## 7. Save the optimized submission

In [18]:
# Detailed file
detail_cols = [
    'id', 'Literal', 'Literal_norm', 'router_pred',
    'exact_code', 'fuzzy_code', 'fuzzy_score',
    'retrieval_code', 'retrieval_score',
    'clf_code', 'clf_confidence',
    'final_code', 'final_method', 'final_confidence'
]
test[detail_cols].to_csv('final_predictions_optimized.csv', index=False)

# Clean submission
submission = test[['id', 'final_code']].copy()
submission.columns = ['id', 'Code']
submission.to_csv('kaggle_submission_optimized.csv', index=False)

print('Saved:')
print('  final_predictions_optimized.csv (full detail)')
print('  kaggle_submission_optimized.csv (Kaggle submission)')
print()
print('Submission sample:')
submission.head(10)

Saved:
  final_predictions_optimized.csv (full detail)
  kaggle_submission_optimized.csv (Kaggle submission)

Submission sample:


,id,Code
0,1,10903ZU
1,2,E210
2,3,G43909
3,4,B182
4,5,N611
5,6,O3413
6,7,E915
7,8,8500
8,9,Q780
9,10,R011


## 8. Summary of improvements

The optimized pipeline achieves about 39% validation accuracy compared to 38.1% with the original Step 3 cascade. The improvements come from:

1. **Stronger classifier**: switching to Linear SVM with C=2.0 and including codes with as few as 2 examples brought classifier accuracy from 27% to 33% in isolation.

2. **Smarter ensemble**: instead of a strict cascade where Step 1 always wins, we weight each method's confidence and pick per-row. This lets the classifier override low-confidence fuzzy matches.

3. **Method weighting**: classifier gets 1.5x weight because its high-confidence predictions are most reliable.

The theoretical ceiling (if we always picked the right method for each literal) is around 49-50%. The gap between our 39% and that ceiling represents cases where our confidence signal isn't reliable enough to pick the right method.... this is the main direction for further improvement, but reaching higher would likely require a deep learning model or more training data.